In [1]:
import anndata as ad
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np

import warnings
warnings. filterwarnings('ignore')

In [2]:
from utils import get_scgpt_model, binning

In [13]:
# bulk_data_path = '../data/GTEx/processed/all_gtex_processed.csv'
bulk_data_path = '../data/GTEx/processed/gene_tpm_v11_bladder_processed.csv'

In [14]:
df_bladder = pd.read_csv(bulk_data_path, index_col=0)
df = pd.read_csv(bulk_data_path, index_col=0)

In [ ]:

adata = ad.AnnData(X=df.values)
adata.obs_names = df.index
adata.var_names = df.columns

In [8]:
display(df)

,OR4F5,OR4F29,OR4F16,SAMD11,NOC2L,KLHL17,PLEKHN1,PERM1,HES4,ISG15,...,LDLRAD3,COMMD9,PRR5L,TRAF6,RAG1,RAG2,IFTAP,LRRC4C,API5,TTC17
GTEX-1117F-0005-SM-HL9SH,0.000000,0.086039,0.000000,0.051557,2.488944,1.207556,0.333340,0.111372,1.255265,3.200660,...,1.008881,2.112847,1.081612,0.899962,0.022584,0.0,0.044769,0.000000,1.980555,2.133610
GTEX-1117F-0011-R10b-SM-GI4VE,0.066345,0.177866,0.000000,0.107933,5.018808,2.647448,0.824405,0.137448,3.147028,4.393368,...,2.601818,3.698507,1.308809,2.830464,0.245217,0.0,1.875544,3.226473,5.234425,4.067165
GTEX-1117F-0011-R11b-SM-GIN8R,0.110769,0.045144,0.000000,0.337386,5.236083,4.032427,1.151346,0.087146,3.021003,2.961303,...,4.327199,3.242615,1.010816,2.916519,0.063340,0.0,1.281769,1.156364,5.257320,4.887009
GTEX-1117F-0011-R2b-SM-GI4VL,0.092739,0.065265,0.065265,0.132110,3.344138,0.728920,0.034874,0.042950,3.401662,4.171591,...,2.352521,3.307238,2.010010,1.701837,0.199763,0.0,0.936437,2.151878,4.039007,2.791722
GTEX-1117F-0011-R3a-SM-GJ3PJ,0.048212,0.211528,0.088366,0.212805,5.027309,2.336032,0.564314,0.194543,4.169548,4.736562,...,1.989909,3.912602,0.586922,2.178810,0.181150,0.0,1.552918,3.051758,4.592110,3.522571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GTEX-ZZPU-2326-SM-GOQYU,0.059017,0.000000,0.000000,1.991071,5.893241,3.254700,0.616785,0.410798,5.853769,5.372580,...,4.800838,3.546549,1.025348,3.255922,0.219571,0.0,1.817107,3.131356,5.629517,5.345556
GTEX-ZZPU-2426-SM-5E44I,0.172001,0.265112,0.000000,2.862017,5.973545,3.251792,0.713646,1.191777,7.489262,5.790905,...,1.821282,3.112136,0.297542,3.303386,0.107937,0.0,1.705182,0.318657,6.020927,4.779796
GTEX-ZZPU-2526-SM-GOQZ3,0.017215,0.047494,0.000000,0.212282,6.401929,4.453340,6.397601,2.789013,4.993321,3.842939,...,3.995322,4.025055,1.055376,3.255727,0.412298,0.0,0.893959,0.606401,5.343717,5.079724
GTEX-ZZPU-2626-SM-5E45Y,0.300605,0.155723,0.000000,1.056389,6.326607,2.036823,0.416618,6.140885,1.780545,2.319788,...,3.724705,3.543657,0.307152,2.612876,0.147040,0.0,1.396686,0.499570,5.265336,3.693242


In [ ]:
# df_small = df.head()

In [19]:
columns_bladder = set(df_bladder.columns)
columns_all_processed = set(df_small.columns)

In [ ]:
len(columns_all_processed), len(columns_bladder & columns_all_processed) # all in processed are present in bladder

(10362, 10362)

In [12]:
df_small.to_csv('../data/GTEx/processed/all_gtex_sample.csv')

In [7]:
model_path = "../papers/scgpt/save/whole_human"

In [8]:
scgpt_model, vocab = get_scgpt_model(model_path, device='cpu', eval=False, do_mvc=False)

In [23]:
# Filter genes that are in the vocabulary
genes_in_vocab = [g for g in adata.var_names if g in vocab]
adata = adata[:, genes_in_vocab].copy()

# Tokenize gene names to IDs
gene_ids = vocab(adata.var_names.tolist())

In [24]:
adata.X = binning(adata.X, n_bins=51)

In [25]:
adata.X

array([[ 3.,  0.,  2., ..., 49., 49., 49.],
       [ 1.,  0.,  0., ..., 49., 49., 49.],
       [ 1.,  2.,  1., ..., 49., 49., 49.],
       ...,
       [ 2.,  5.,  0., ..., 49., 49., 49.],
       [ 2.,  0.,  0., ..., 49., 49., 49.],
       [ 1.,  5.,  3., ..., 49., 49., 49.]])

In [26]:
def mask_expression(values, mask_ratio=0.15):
    """
    Randomly mask gene expression values.
    values: binned expression tensor
    """
    prob_matrix = torch.full(values.shape, mask_ratio)
    mask = torch.bernoulli(prob_matrix).bool()
    
    masked_values = values.clone()
    masked_values[mask] = 0  # Replace masked genes with 0 (bin for zero/masked)
    
    return masked_values, mask

In [27]:

def masked_mse_loss(predicts, targets, mask):
    """
    Only compute loss for masked positions.
    predicts: output from model.expr_decoder
    targets: original binned values
    mask: boolean mask from step 1
    """
    loss = F.mse_loss(predicts[mask], targets[mask].float())
    return loss

In [28]:
class SCDataset(Dataset):
    def __init__(self, data_matrix, gene_ids, batch_labels):
        """
        data_matrix: Macierz ekspresji (po binningu), kształt [komórki, geny]
        gene_ids: Lista ID genów ze słownika (vocab), kształt [geny]
        batch_labels: ID partii dla każdej komórki, kształt [komórki]
        """
        self.data = torch.tensor(data_matrix, dtype=torch.float32)
        self.gene_ids = torch.tensor(gene_ids, dtype=torch.long)
        self.batch_labels = torch.tensor(batch_labels, dtype=torch.long)

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        # Pobieramy wartości ekspresji dla danej komórki
        values = self.data[idx]
        
        # W scGPT zazwyczaj podajemy tylko geny, które mają ekspresję > 0 
        # lub stosujemy stałą listę genów (HVG).
        # Tutaj przykład dla stałej listy genów:
        src = self.gene_ids
        
        # Maska paddingu (jeśli wszystkie komórki mają te same geny, maska to same False)
        mask = torch.zeros_like(src, dtype=torch.bool)
        
        return src, values, mask, self.batch_labels[idx]


In [36]:
binned_counts = adata.X
vocab_gene_ids = gene_ids
batch_indices = np.zeros(adata.shape[0], dtype=np.int64)

dataset = SCDataset(binned_counts, vocab_gene_ids, batch_indices)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

In [39]:
# Przygotowanie urządzenia (GPU jeśli dostępne, inaczej CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scgpt_model.to(device)

scgpt_model.train()
optimizer = torch.optim.Adam(scgpt_model.parameters(), lr=1e-4)

for src, values, pad_mask, batch_idx in dataloader:
    optimizer.zero_grad()
    
    # Przeniesienie danych na odpowiednie urządzenie
    src = src.to(device)
    values = values.to(device)
    pad_mask = pad_mask.to(device)
    batch_idx = batch_idx.to(device)
    
    # --- MASKOWANIE (MLM Task) ---
    # values to nasze oryginalne targey, masked_values podajemy do modelu
    masked_values, bool_mask = mask_expression(values, mask_ratio=0.15)
    
    # --- FORWARD PASS ---
    # UWAGA: Podajemy masked_values jako wejście do modelu!
    results = scgpt_model(
        src=src, 
        values=masked_values, 
        src_key_padding_mask=pad_mask, 
        # batch_labels=batch_idx,
        CLS=False, 
        CCE=False,
        MVC=False,
        ECS=False
    )
    
    # Przewidziane wartości ekspresji
    preds = results["mlm_output"]
    
    # --- OBLICZANIE STRATY ---
    # Porównujemy predykcje z ORYGINALNYMI values, tylko na pozycjach maskowanych
    loss = masked_mse_loss(preds, values, bool_mask)
    
    loss.backward()
    optimizer.step()
    
    # Opcjonalnie: print(f"Loss: {loss.item():.4f}")

OutOfMemoryError: CUDA out of memory. Tried to allocate 612.00 MiB. GPU 0 has a total capacity of 47.40 GiB of which 585.69 MiB is free. Including non-PyTorch memory, this process has 46.82 GiB memory in use. Of the allocated memory 46.43 GiB is allocated by PyTorch, and 90.26 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
torch.cuda.empty_cache()

: 

In [9]:
print(vocab['<pad>'])

60694
